> **STATUS — UNEXECUTED PROTOCOL. Requires an external Replogle h5ad file.**
>
> This notebook mirrors `01_measure_k562.ipynb` but consumes a **processed
> Replogle 2022 h5ad** instead of the raw 10x tarballs used by `01`. On the
> Replogle essential-gene screens the tool auto-routes the efficiency
> estimator to `detection_rate` because the shipped h5ad is pre-scaled
> z-score residuals (see MANUSCRIPT.md §4.3 and Fig. S2), reaches full-rank
> identification, but the observed linearity statistic sits at the noise-floor
> — MANUSCRIPT.md §2.6 shows that a matched-scale synthetic *linear* ground
> truth produces the same statistic value at this (d, n, U, κ, σ), so the
> observation is not evidence against linearity but reveals the diagnostic
> is noise-limited at published scale.
>
> **Download prerequisite.** From `https://gwps.wi.mit.edu`, fetch one of:
> - `K562_essential_normalized_singlecell_01.h5ad` (K562 essential-gene screen)
> - `RPE1_normalized_singlecell_01.h5ad` (RPE1 essential-gene screen)
>
> Set `SCJDO_K562_DATA_ROOT` to point at the containing directory.

# 01b — Measure a K562 essential-gene Perturb-seq operator (Replogle 2022)

## 1. Configuration

In [1]:
import os
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import anndata as ad

import anchorop as ao

DATA_ROOT = Path(
    os.environ.get("SCJDO_K562_DATA_ROOT", "")
    or "/Users/terooatt/Downloads/anchor-op-source/examples/data"
)
H5AD_FILENAME = "K562_essential_normalized_singlecell_01.h5ad"  # adjust to match your download

H5AD_PATH = DATA_ROOT / H5AD_FILENAME
assert H5AD_PATH.exists(), f"h5ad not found at {H5AD_PATH}. Download from gwps.wi.mit.edu."

MIN_CELLS_PER_TARGET = 60
MIN_CELLS_PER_GUIDE = 30
MIN_KD_EFFICIENCY = 0.05
D_PROGRAMS = 30
N_HVG = 3000
N_TARGETS_KEEP = 200          # essential-gene screens have ~1000+ targets; a subset for tractability
AGGREGATE_TO_TARGET = True
SEED = 20260729

DATA_PROVENANCE = {
    "source_path": str(H5AD_PATH),
    "seed": SEED,
    "min_cells_per_target": MIN_CELLS_PER_TARGET,
    "min_cells_per_guide": MIN_CELLS_PER_GUIDE,
    "min_knockdown_efficiency": MIN_KD_EFFICIENCY,
    "d_programs": D_PROGRAMS,
    "n_hvg": N_HVG,
    "n_targets_keep": N_TARGETS_KEEP,
    "aggregate_to_target": AGGREGATE_TO_TARGET,
    "rank_tol": 1e-2,
    "efficiency_estimator": "auto",
}
print(DATA_PROVENANCE)

{'source_path': '/Users/terooatt/Downloads/anchor-op-source/examples/data/K562_essential_normalized_singlecell_01.h5ad', 'seed': 20260729, 'min_cells_per_target': 60, 'min_cells_per_guide': 30, 'min_knockdown_efficiency': 0.05, 'd_programs': 30, 'n_hvg': 3000, 'n_targets_keep': 200, 'aggregate_to_target': True, 'rank_tol': 0.01, 'efficiency_estimator': 'auto'}


## 2. Load with the Replogle-aware loader

`ao.load_replogle_h5ad` auto-detects the target/guide/batch column names
(they vary across Replogle releases) and returns an AnnData whose `obs`
already has canonical `guide` and `target_gene` columns.

In [2]:
adata = ao.load_replogle_h5ad(str(H5AD_PATH))
print("provenance:", adata.uns["anchorop_replogle_provenance"])
print()
print(f"loaded {adata.shape}")
print(f"NT cells: {(adata.obs['target_gene'] == '').sum()}")
print(f"unique targets (perturbed): {(adata.obs['target_gene'] != '').groupby(adata.obs['target_gene']).ngroup().max() + 1}")

provenance: {'source_path': '/Users/terooatt/Downloads/anchor-op-source/examples/data/K562_essential_normalized_singlecell_01.h5ad', 'detected_target_col': 'gene', 'detected_guide_col': 'sgID_AB', 'detected_batch_col': 'gem_group', 'detected_control_label': 'non-targeting', 'used_gene_id_for_target': True, 'var_names_look_like_ensembl': True, 'n_cells': 310385, 'n_control_cells': 10691, 'n_perturbed_cells': 299694, 'n_unique_targets': 2057, 'n_unique_guides': 2176, 'canonical_control_label': 'non-targeting', 'canonical_guide_key': 'guide', 'canonical_target_key': 'target_gene', 'backed': None}

loaded (310385, 8563)
NT cells: 10691
unique targets (perturbed): 2058


## 3. Choose targets and optionally aggregate to target level

In [3]:
target_counts = adata.obs.loc[adata.obs["target_gene"] != "", "target_gene"].value_counts()
qualifying = target_counts[target_counts >= MIN_CELLS_PER_TARGET].index.tolist()
if N_TARGETS_KEEP is not None:
    qualifying = qualifying[:N_TARGETS_KEEP]
keep_mask = adata.obs["target_gene"].isin(qualifying) | (adata.obs["target_gene"] == "")
adata = adata[keep_mask].copy()

if AGGREGATE_TO_TARGET:
    perturbed = adata.obs["target_gene"] != ""
    adata.obs.loc[perturbed, "guide"] = "guide_" + adata.obs.loc[perturbed, "target_gene"].astype(str)

print(f"after target selection: {adata.shape}, {len(qualifying)} targets kept")

after target selection: (88556, 8563), 200 targets kept


## 4. QC + normalize + HVG ∪ target genes

The Replogle processed h5ads are typically already log-normalized, so `sc.pp.normalize_total` + `log1p`
may be redundant — check the source h5ad and skip normalization if the data are
already on the log scale.

In [4]:
import scanpy as sc

# Replogle 'normalized_singlecell' files are per-gene z-score / Pearson-residual
# scaled — not log-normalized counts. RPE1 also has some NaN/inf entries (a few
# genes with numerical issues). Filter those before detection + HVG.
sample = adata.X[:2000, :]
if hasattr(sample, 'toarray'):
    sample = sample.toarray()
sample = np.asarray(sample)
finite_col = np.isfinite(sample).all(axis=0)
if not finite_col.all():
    print(f"filtering out {(~finite_col).sum()} genes with non-finite values in first 2000-cell sample")
    keep_mask = np.isfinite(np.asarray(adata.X[:min(20000, adata.n_obs), :].toarray() if hasattr(adata.X, 'toarray') else adata.X[:min(20000, adata.n_obs), :])).all(axis=0)
    adata = adata[:, keep_mask].copy()
    print(f"after finite-gene filter: {adata.shape}")
    sample = adata.X[:2000, :]
    if hasattr(sample, 'toarray'):
        sample = sample.toarray()
    sample = np.asarray(sample)

is_prescaled = (np.nanmin(sample) < -1) or (abs(np.nanmean(sample)) < 0.1 and np.nanmax(sample) > 10)
print(f"data range: [{np.nanmin(sample):.2f}, {np.nanmax(sample):.2f}], mean {np.nanmean(sample):.3f}")
print(f"detected as pre-scaled (skip normalize + use variance-based HVG): {is_prescaled}")

if not is_prescaled:
    # Standard raw-count path.
    adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)
    adata = adata[(adata.obs["pct_counts_mt"] < 20) & (adata.obs["total_counts"] > 500)].copy()
    if float(adata.X.mean()) > 5.0:
        adata.layers["raw_counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    sc.pp.filter_genes(adata, min_cells=3)
    sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat")
else:
    # Pre-scaled path: skip normalize/filter_genes; pick HVG by variance.
    print("computing per-gene variance for HVG selection ...")
    X = adata.X
    n_chunks = 20
    chunk_size = X.shape[0] // n_chunks + 1
    sums = np.zeros(X.shape[1], dtype=np.float64)
    sums_sq = np.zeros(X.shape[1], dtype=np.float64)
    counts = np.zeros(X.shape[1], dtype=np.float64)
    for k in range(n_chunks):
        block = X[k*chunk_size:(k+1)*chunk_size, :]
        block = block.toarray() if hasattr(block, 'toarray') else np.asarray(block)
        finite = np.isfinite(block)
        block_clean = np.where(finite, block, 0.0)
        sums += block_clean.sum(axis=0)
        sums_sq += (block_clean**2).sum(axis=0)
        counts += finite.sum(axis=0)
    counts = np.maximum(counts, 1.0)
    gene_mean = sums / counts
    gene_var = (sums_sq / counts) - gene_mean**2
    gene_var = np.asarray(gene_var).ravel()
    gene_var = np.where(np.isfinite(gene_var), gene_var, 0.0)
    top_idx = np.argsort(gene_var)[::-1][:N_HVG]
    adata.var["highly_variable"] = False
    adata.var.iloc[top_idx, adata.var.columns.get_loc("highly_variable")] = True
    print(f"selected top {N_HVG} genes by variance")

target_symbols = {
    t for t in adata.obs["target_gene"].astype(str).unique() if t and t != "non-targeting"
}
in_matrix = target_symbols & set(adata.var_names)
retain_mask = adata.var["highly_variable"].to_numpy() | adata.var_names.isin(in_matrix)
adata_hvg = adata[:, retain_mask].copy()
print(f"HVG ∪ targets: {adata_hvg.shape}")


data range: [-5.56, 494.38], mean 0.023
detected as pre-scaled (skip normalize + use variance-based HVG): True
computing per-gene variance for HVG selection ...
selected top 3000 genes by variance
HVG ∪ targets: (88556, 3142)


## 5. Fit control-only program basis

In [5]:
# For Replogle pre-scaled data (has negative values), NMF is not applicable.
# Use PCA on controls and wrap the loadings as a program basis. For raw-count
# data, ao.fit_programs (NMF/cNMF) is the correct default — see 01_measure_k562.
from sklearn.decomposition import PCA

control_mask = (adata_hvg.obs["target_gene"] == "").to_numpy()
print(f"control cells for basis fit: {int(control_mask.sum())}")
X_ctrl = adata_hvg[control_mask].X
if hasattr(X_ctrl, 'toarray'):
    X_ctrl = X_ctrl.toarray()
X_ctrl = np.asarray(X_ctrl, dtype=np.float32)
pca = PCA(n_components=D_PROGRAMS, random_state=SEED)
pca.fit(X_ctrl)
# PCA components_: (d, n_genes); we want gene-by-program loadings.
loadings = pca.components_.T.astype(np.float32)  # (n_genes, d)
basis = ao.make_program_basis(
    loadings, adata_hvg.var_names,
    method="pca_external", control_count=int(control_mask.sum()),
    normalize=False,
    metadata={
        "pca_explained_variance_ratio_sum": float(pca.explained_variance_ratio_.sum()),
        "pca_singular_values_first_last": (float(pca.singular_values_[0]), float(pca.singular_values_[-1])),
    },
)
print({"d": basis.d, "n_genes": basis.n_genes,
       "cum_explained_variance": round(basis.metadata["pca_explained_variance_ratio_sum"], 3)})


control cells for basis fit: 10691
{'d': 30, 'n_genes': 3142, 'cum_explained_variance': 0.068}


## 6. Measure the operator

Uses the current defaults (`rank_tol=1e-2`, `efficiency_estimator="auto"`). Because this Replogle h5ad ships as pre-scaled z-score residuals (mean≈0 controls, ~68% negative values), the `auto` router selects `detection_rate` as the signed distributional-shift statistic valid on this data class — `mean_ratio` is undefined at ctrl_mean≈0 (see MANUSCRIPT.md §4.3 and Fig. S2). The resolved choice is recorded in `report.notes` with a `(auto-routed from data format)` suffix for provenance.

In [6]:
measurement = ao.measure_operator(
    adata_hvg, basis,
    guide_key="guide", target_key="target_gene",
    control_label="non-targeting",
    min_cells_per_guide=MIN_CELLS_PER_GUIDE,
    min_knockdown_efficiency=MIN_KD_EFFICIENCY,
    reg="tsvd", reg_param="path", rank_tol=1e-2,
    bootstrap=100, bootstrap_seed=SEED,
    state_label="K562_essential",
)
r = measurement.report
print({
    "full_domain_identified": r.full_domain_identified,
    "effective_response_rank": f"{r.effective_response_rank}/{r.d}",
    "condition_number": round(r.condition_number, 2),
    "retained_guides": f"{len(r.retained_guides)}/{r.n_guides_input}",
})

AnchorOpError: The response matrix appears signed normalized/residual-like and cannot calibrate κ. Provide an exactly paired raw/count-like calibration_adata, or set allow_proxy_efficiency=True only for explicitly uncalibrated exploratory outputs.

## 7. Standard diagnostic + linearity check

In [ ]:
report = ao.analyses.measurement_report(measurement)
report["figures"]["diagnostics"]

In [ ]:
report["figures"]["guide_drops"]

In [ ]:
linearity = ao.linearity_check(measurement, threshold=0.25, n_null=200, null_seed=42)
print({
    "passed_preregistered_raw_criterion": linearity.passed,
    "relative_difference": round(linearity.relative_difference, 3),
    "overlap_rank": linearity.overlap_rank,
    "null_median": round(linearity.null_median, 3) if linearity.null_median is not None else None,
    "null_std": round(linearity.null_std, 4) if linearity.null_std is not None else None,
    "null_p95": round(linearity.null_p95, 3) if linearity.null_p95 is not None else None,
    "excess_above_null": round(linearity.excess_above_null, 3) if linearity.excess_above_null is not None else None,
    "z_score": round(linearity.z_score, 2) if linearity.z_score is not None else None,
    "n_null": linearity.n_null,
})
if not linearity.passed:
    print()
    print("Preregistered raw rel_diff > 0.25 threshold not reached — but this is not evidence against linearity.")
    print("Per MANUSCRIPT.md §2.6, the diagnostic is noise-limited at Replogle scale:")
    print(f"  Observed rel_diff = {linearity.relative_difference:.3f}")
    print("  Matched-scale synthetic *linear* ground truth at K562 σ ≈ 0.24 gives rel_diff ≈ 1.51")
    # i.e. a perfectly linear system at the same (d, n, U, κ, σ) reproduces the observed value
    # within ~0.05, so the raw statistic cannot distinguish linear from nonlinear at this scale.
    print(f"  Random-split null (empirical) median = {linearity.null_median:.3f}, std = {linearity.null_std:.4f}")
    print(f"  Excess above null = {linearity.excess_above_null:+.3f}, z-score = {linearity.z_score:+.2f}")
    print("  The preregistered 0.25 threshold was unreachable at (d=30, n≈200, σ≈0.24)")
    print("  by any dataset, linear or not. See MANUSCRIPT.md §2.6 (linearity power analysis)")
    print("  and §3.1 (discussion). Design conditions for detection power are in §3.4.")

## 8. Save the bundle for downstream notebooks

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
BUNDLE = RESULTS_DIR / "k562_essential_measurement.pkl"
with BUNDLE.open("wb") as f:
    pickle.dump({"measurement": measurement, "basis": basis, "linearity": linearity,
                 "provenance": DATA_PROVENANCE}, f)
print("saved measurement bundle to", BUNDLE)
print("02_benchmark.ipynb and 03_archetypes.ipynb pick up any *_measurement.pkl automatically.")